# Notebook 8 - End-to-End Demonstration and Evaluation

This notebook executes the fixed roadmap scenario through ASR, the frozen trained intent model, grounding, planning, scheduling, safety, deterministic software simulation, a disjoint Agriculture-Vision holdout, reporting, and the completion gate. Missing evidence stops the run; it is never replaced with a zero or cached demo value.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
REPO = Path('/content/shepherd-ai') if IN_COLAB else Path.cwd()
if IN_COLAB and not (REPO / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/week8-end-to-end', 'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO)], check=True)
if not (REPO / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from the Shepherd-AI repository.')
%cd {REPO}
%pip install -q -e '.[vision]' openai-whisper


## 1. Runtime and private research storage

The fixed vision dataset and large checkpoints remain in private Google Drive. A T4 is required for registered GPU inference; this notebook does not retrain either model.

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Connect an NVIDIA T4 runtime before running model inference.')
PRIVATE_ROOT = Path('/content/drive/MyDrive/shepherd-ai-private')
WEEK8_PRIVATE = PRIVATE_ROOT / 'week8'
WEEK8_PRIVATE.mkdir(parents=True, exist_ok=True)
print({'device': torch.cuda.get_device_name(0), 'private_output': str(WEEK8_PRIVATE)})


## 2. Register and transcribe the exact spoken scenario

Upload one PCM WAV in which you say exactly: **Send two drones north to inspect crops and one drone east to inspect irrigation.** The recording must be human speech.

In [ ]:
from google.colab import files
uploaded = files.upload()
wav_names = [name for name in uploaded if name.lower().endswith('.wav')]
if len(wav_names) != 1:
    raise RuntimeError('Upload exactly one PCM WAV file.')
subprocess.run(['python', 'scripts/register_week8_audio.py', '--wav', wav_names[0]], check=True)
subprocess.run([
    'python', 'scripts/transcribe_audio_whisper.py',
    '--manifest', 'datasets/sample_audio/week8_exact_scenario_manifest.jsonl',
    '--dataset-root', '.',
    '--predictions-output', 'outputs/evaluations/week8_exact_scenario_whisper_predictions.jsonl',
    '--evaluation-output', 'outputs/evaluations/week8_exact_scenario_whisper_evaluation.json',
    '--model', 'base', '--device', 'cuda', '--language', 'en', '--fp16',
    '--required-device-substring', 'T4',
], check=True)
subprocess.run([
    'python', 'scripts/build_week8_asr_evidence.py',
    '--manifest', 'datasets/sample_audio/week8_exact_scenario_manifest.jsonl',
    '--predictions', 'outputs/evaluations/week8_exact_scenario_whisper_predictions.jsonl',
    '--evaluation', 'outputs/evaluations/week8_exact_scenario_whisper_evaluation.json',
], check=True)
json.loads(Path('outputs/evaluations/week8_exact_scenario_asr.json').read_text())


## 3. Run the frozen trained intent checkpoint

The first run may ask for the previously exported `hf_token_classifier_distilbert_colab_t4_expanded85.zip`. It is extracted once into private Drive and reused on later sessions.

In [ ]:
NLP_ARTIFACT_ROOT = PRIVATE_ROOT / 'week2/model_artifacts'
NLP_MODEL_DIR = NLP_ARTIFACT_ROOT / 'hf_token_classifier_distilbert_colab_t4_expanded85'
if not (NLP_MODEL_DIR / 'model.safetensors').exists():
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError('Upload exactly the expanded85 checkpoint zip.')
    subprocess.run([
        'python', 'scripts/import_hf_checkpoint.py', '--checkpoint-zip', zip_names[0],
        '--output-dir', str(NLP_ARTIFACT_ROOT),
        '--expected-name', 'hf_token_classifier_distilbert_colab_t4_expanded85', '--overwrite',
    ], check=True)
subprocess.run([
    'python', 'scripts/predict_hf_token_classifier_transcripts.py',
    '--input-jsonl', 'datasets/commands/week8_fixed_scenario_clauses.jsonl',
    '--model-dir', str(NLP_MODEL_DIR),
    '--output', 'outputs/evaluations/week8_fixed_clause_hf_predictions.json',
    '--transcript-field', 'text', '--batch-size', '2',
    '--required-device-substring', 'T4',
], check=True)
subprocess.run([
    'python', 'scripts/evaluate_week8_intent.py',
    '--predictions', 'outputs/evaluations/week8_fixed_clause_hf_predictions.json',
    '--model-path', str(NLP_MODEL_DIR / 'model.safetensors'),
], check=True)
json.loads(Path('outputs/evaluations/week8_exact_scenario_intent_evaluation.json').read_text())


## 4. Ground, plan, schedule, validate, and simulate

The operator-confirmed destination for clause 2 remains East Field; irrigation remains the semantic inspection target.

In [ ]:
subprocess.run([
    'python', 'scripts/run_week8_simulation.py',
    '--output', 'outputs/evaluations/week8_roadmap_scenario_simulation.json',
    '--telemetry-output', 'outputs/evaluations/week8_roadmap_scenario_telemetry.jsonl',
    '--map-output', 'outputs/visualizations/week8_roadmap_scenario_simulation.html',
], check=True)


## 5. Evaluate mission-assigned agricultural imagery

This uses the official validation remainder after excluding every fixed Week 6 development ID. Crop inspection uses non-water anomaly classes; irrigation uses `water` and `waterway`. The mapping is project-defined and the result is not the official hidden-test benchmark.

In [ ]:
WEEK6_ROOT = PRIVATE_ROOT / 'week6'
DATASET_DIR = WEEK6_ROOT / 'agriculture-vision'
EXTRACTED = DATASET_DIR / 'data2017'
SPLITS = DATASET_DIR / 'data2017_splits.json'
LABELS_DIR = EXTRACTED / 'data2017_miniscale'
VISION_CHECKPOINT = WEEK6_ROOT / 'checkpoints/agriculture_vision_small_unet_seed17_dev256_bce_dice1/best.pt'
for required in (EXTRACTED, SPLITS, LABELS_DIR, VISION_CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(f'Run Notebook6_Vision setup/training first; missing {required}')
FULL_MANIFEST = Path('datasets/aerial_images/manifest_week8_full.jsonl')
DEV_MANIFEST = Path('datasets/aerial_images/manifest_week8_dev256.jsonl')
for output, maximum in ((FULL_MANIFEST, '10000'), (DEV_MANIFEST, '256')):
    if output.exists():
        print(f'Reusing prepared manifest: {output}')
        continue
    subprocess.run([
        'python', 'scripts/prepare_agriculture_vision_subset.py',
        '--dataset-dir', str(EXTRACTED), '--dataset-root', str(WEEK6_ROOT),
        '--split-json', str(SPLITS), '--output', str(output),
        '--max-per-split', maximum, '--selection-strategy', 'seeded-hash',
        '--selection-seed', '17', '--accept-terms',
    ], check=True)
subprocess.run([
    'python', 'scripts/evaluate_week8_agriculture_vision.py',
    '--full-manifest', str(FULL_MANIFEST), '--development-manifest', str(DEV_MANIFEST),
    '--dataset-root', str(WEEK6_ROOT), '--labels-dir', str(LABELS_DIR),
    '--checkpoint', str(VISION_CHECKPOINT),
    '--mission-manifest-output', 'outputs/evaluations/week8_mission_image_manifest.jsonl',
    '--raw-predictions-output', 'outputs/evaluations/week8_mission_vision_raw.jsonl',
    '--evaluation-output', 'outputs/evaluations/week8_mission_vision_evaluation.json',
    '--max-per-clause', '32', '--selection-seed', '29', '--batch-size', '4',
    '--device', 'cuda', '--required-device-substring', 'T4',
], check=True)


## 6. Build the five roadmap metrics and final artifacts

In [ ]:
subprocess.run([
    'python', 'scripts/build_week8_evaluation.py',
    '--simulation', 'outputs/evaluations/week8_roadmap_scenario_simulation.json',
    '--asr-evidence', 'outputs/evaluations/week8_exact_scenario_asr.json',
    '--intent-evaluation', 'outputs/evaluations/week8_exact_scenario_intent_evaluation.json',
    '--vision-evaluation', 'outputs/evaluations/week8_mission_vision_evaluation.json',
], check=True)
subprocess.run([
    'python', 'scripts/render_week8_screenshot.py',
    '--simulation', 'outputs/evaluations/week8_roadmap_scenario_simulation.json',
    '--telemetry', 'outputs/evaluations/week8_roadmap_scenario_telemetry.jsonl',
    '--map', 'datasets/maps/shepherd_test_map_v1.csv',
], check=True)
subprocess.run(['python', 'scripts/audit_week8_completion.py'], check=True)
audit = json.loads(Path('outputs/evaluations/week8_completion_gate_audit.json').read_text())
if not audit['completion_allowed']:
    raise RuntimeError(f"Week 8 remains incomplete: {audit['blockers']}")
audit


## 7. Persist raw and derived artifacts

The bundle is copied to private Drive before download. Licensed pixels and model weights are not included.

In [ ]:
ARTIFACT_ROOT = Path('/content/week8_completed_artifacts')
if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
for source in (Path('outputs/evaluations'), Path('outputs/visualizations'), Path('reports')):
    destination = ARTIFACT_ROOT / source
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination)
archive = Path(shutil.make_archive('/content/week8_completed_artifacts', 'zip', ARTIFACT_ROOT))
drive_copy = WEEK8_PRIVATE / archive.name
shutil.copy2(archive, drive_copy)
print({'private_drive_copy': str(drive_copy), 'size_bytes': archive.stat().st_size})
files.download(str(archive))
